# Object detection 3: Faster R-CNN on our MobileNetV3

Builds a detector around **our own classifier**. A detector has three parts:

- **backbone**: turns the image into feature maps. Here it is the MobileNetV3 from our classifier's `best.pth` (everything except its 10-class head);
- **neck**: a feature pyramid (FPN) that combines the backbone's maps at three scales, so small and big objects are both seen at a useful resolution;
- **heads**: a region proposal network suggests boxes that may hold an object, and a box head names each one (10 classes + background) and refines its edges (torchvision's Faster R-CNN).

The backbone already knows *what* our objects look like; the neck and heads start from scratch and learn *where* from the boxes in `data_object/`. Training uses flips and mosaic (four photos in one image), because our photos nearly always show one big object in the middle. The best epoch is picked on the val split; the test split is scored once at the end with the same code as notebook 02.

**On Kaggle:** attach the `data_object` dataset made by notebook 01 with *Add Input*, and turn on a GPU and *Internet* in the side panel (the setup cell downloads the project code from GitHub, so it runs the **pushed** version of `imgcls/`). Runs are saved in `/kaggle/working/output`. To start from our classifier there, also attach a dataset holding its run folder (`best.pth` with `summary.json`); otherwise ImageNet weights are used.

Generated by `scripts/create_detection_notebooks.py` on 2026-11-24. To reset this notebook: `python scripts/create_detection_notebooks.py --only 03 --force`

In [ ]:
# ---- settings for this model (edit freely) ----
MODEL_NAME = 'frcnn_mobilenetv3'  # results go to output/<MODEL_NAME>/<timestamp>/
ARCH = 'mobilenetv3_large_100.ra_in1k'  # timm backbone; must match the classifier checkpoint
BACKBONE_CHECKPOINT = 'auto'    # "auto" = newest output/<run>/best.pth of a classifier with ARCH (Kaggle: an attached run folder); a path; None = ImageNet weights
IMG_SIZE = 512                  # longer side of the training images
EPOCHS = 20
BATCH_SIZE = 8                  # lower it if the GPU runs out of memory (4 for a 4 GB GPU)
LR = 2e-4                       # peak learning rate (AdamW, 1 warm-up epoch, cosine)
WEIGHT_DECAY = 1e-4
MOSAIC = 0.5                    # share of training images built from 4 photos (small and several objects per image)
VAL_LIMIT = None                # validate on at most this many images per epoch (CPU: e.g. 200); None = all
NUM_WORKERS = 2                 # image-loading processes; 0 if loading hangs on Windows
DEVICE = 'auto'                 # "auto" = GPU if available, otherwise CPU
CONF = 0.5                      # minimum score for the boxes drawn on pictures and videos (not used for mAP)
VIDEO_PATH = None               # a video to run the detector on, e.g. r"C:\Users\me\Downloads\clip.mp4"
VIDEO_FPS = 2.0                 # frames analysed per second of video
OBJECT_DIR = None               # None = find data_object/ (project folder; Kaggle: attached dataset)
SEED = 42

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

# find the project (the folder holding imgcls/); on Colab / Kaggle fetch it from GitHub first
REPO_URL = "https://github.com/M0HiT27/Image-Classification.git"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imgcls").is_dir()), None)
if ROOT is None:
    on_colab = Path("/content").is_dir() and not Path("/kaggle").is_dir()  # Kaggle has /content too
    ROOT = Path("/content" if on_colab else "/tmp") / "Image-Classification"
    if (ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))
# forget an imgcls loaded earlier in this session, so re-running this cell after a pull uses the new code
for name in [m for m in sys.modules if m == "imgcls" or m.startswith("imgcls.")]:
    del sys.modules[name]


def ensure(module, pip_name=None):
    """Install a package that is missing (Kaggle and Colab have most of them already)."""
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)


ensure('timm')
ensure('cv2', 'opencv-python-headless')

from IPython.display import Image, display
import imgcls
from imgcls import objdet

from imgcls.objdet import dataset, frcnn, viz

env, object_dir = objdet.setup(OBJECT_DIR)
imgcls.set_seed(SEED)
device = imgcls.get_device(DEVICE)

## Model
Loads our classifier's weights into the backbone (only its classification head is left out) and adds a new feature pyramid and detection heads.

In [ ]:
checkpoint = frcnn.find_backbone_checkpoint(ARCH) if BACKBONE_CHECKPOINT == "auto" else BACKBONE_CHECKPOINT
if checkpoint is None:
    print("No classifier checkpoint for this architecture found: the backbone starts from ImageNet weights.")
model, meta = frcnn.build_model(len(objdet.CLASSES), ARCH, IMG_SIZE, backbone_checkpoint=checkpoint)

## Train
One line per epoch: training loss and validation mAP. `best.pth` is saved whenever val mAP50-95 improves; interrupting the cell keeps it.

In [ ]:
cfg = imgcls.config_from_globals(globals())
run_dir = frcnn.train(cfg, model, meta, object_dir, device)
if (run_dir / "training_curves.png").exists():
    display(Image(filename=str(run_dir / "training_curves.png")))

## Evaluate on the test split
Same metric as notebook 02: **mAP50** (box overlaps the truth by >= 50%) and **mAP50-95** (also rewards tight boxes).

In [ ]:
# to evaluate an earlier run instead: run_dir = imgcls.OUTPUT_DIR / MODEL_NAME / "<timestamp>"
display(frcnn.evaluate_run(run_dir, object_dir, device, "test"))

## Look at predictions
Random test images: coloured boxes are predictions (score >= `CONF`), thin white ones the labels.

In [ ]:
best, ckpt = frcnn.load_checkpoint(run_dir / "best.pth", device)
predict = frcnn.predictor(best, ckpt["img_size"], device, CONF)
grid = viz.predictions_grid(dataset.read_split(object_dir, "test"), predict, run_dir / "test_examples.png")
display(Image(filename=str(grid)))

## Run it on a video
Writes `video_annotated.mp4` and a sheet of frames into the run folder.

In [ ]:
if VIDEO_PATH:
    video, sheet, counts = viz.annotate_video(VIDEO_PATH, predict, run_dir / "video_annotated.mp4", VIDEO_FPS)
    display(Image(filename=str(sheet)))

## Compare detection runs
Every YOLO and Faster R-CNN run in the runs folder, newest first.

In [ ]:
display(objdet.detection_runs())